# Cleaning: Part-2

The goal here is exactly the same as `HW-3.2-cleaning-1.ipynb`, except this time you should repeat the exercise by leveraging an LLM and prompt engineering to streamline the cleaning process. Save the result to `data/processed-jobs-2.csv`.

Essentially, your job is to write an LLM wrapper that extracts structured features from the job descriptions. How you design the prompts, the output schema, and the processing loop is up to you.

## Choose your compute path

This year you have two options for where the LLM runs. Choose one (or try both, if you are curious):

| | Option A: Cloud API | Option B: Local model |
|---|---|---|
| Where it runs | A provider's servers (OpenAI, Google Gemini, Anthropic, etc.) | Your own laptop, via [Ollama](https://ollama.com) |
| Cost model | Pay as you go, per token (some providers have a limited free tier) | Free to run once downloaded; you pay in disk space, RAM, and time |
| Model quality | Frontier models, the strongest available | Smaller open-weight models; good, but expect more mistakes |
| Speed | Fast, but subject to rate limits | Depends on your hardware; can be slow on older laptops |
| Privacy | Your data is sent to a third party | Your data never leaves your machine |
| Setup | An API key and a few lines of code | Install Ollama and download a model (a few GB to 20+ GB) |

Your coding workflow will differ depending on your choice (client setup, model names, speed, rate limits, and how much you can process at once), so adjust your code accordingly. At the top of your submission, state which option you chose, which model you used, and why.

## Why this choice matters

While this is a simple assignment, the choice between local LLM compute and API usage is a serious design decision for real software tools, and one the whole technical community is wrestling with right now.

* **Cloud APIs** give you access to frontier models with no hardware to buy or maintain. You pay as you go, per token, so costs scale with usage, and you depend on the provider's pricing, rate limits, availability, and data policies.
* **On-premise (local) compute** shifts the cost up front: hardware purchase and ongoing maintenance. If you can maintain the infrastructure, the marginal compute cost is much lower, mainly electricity plus the salary of the people who keep the systems running and up to date. You also get full control over your data and no rate limits beyond your own hardware.

Which one wins depends on volume, data sensitivity, required quality, latency, and the team's capacity to run infrastructure. Organizations are making this trade-off right now, and as a data scientist you will be part of these decisions. Use this assignment to get a feel for both sides.

## Option A: Cloud API

You can use any LLM API that you want. For example:

* **OpenAI**: requires buying a few credits, e.g. $5 to $10, which is far more than this assignment needs if you use a small, inexpensive model. Here is an example of how to use OpenAI's API:
  [https://jfh.georgetown.domains/centralized-lecture-content/content/computer-science/general-concepts/openAI-API-example/notes.html](https://jfh.georgetown.domains/centralized-lecture-content/content/computer-science/general-concepts/openAI-API-example/notes.html)
* **Google Gemini**: has a free tier with rate limits. Quickstart (uses the `google-genai` package):
  [https://ai.google.dev/gemini-api/docs/quickstart?lang=python](https://ai.google.dev/gemini-api/docs/quickstart?lang=python)
* Other providers (Anthropic, Mistral, Groq, OpenRouter, etc.) also work. Do some research and find a tool that fits your needs and budget.

Tips for the API path:

* Store your key in `~/.api-keys.json` (as in HW-3.1), never in the notebook or the repo.
* Start with the smallest, cheapest model that does the job well; extraction tasks rarely need a provider's top model.
* Check the provider's rate limits (requests per minute and per day). With roughly 900 job descriptions, you may need to pace requests, or put several job descriptions in one request.
* Keep a rough tally of tokens used and dollars spent; you will report it at the end.

## Option B: Local model with Ollama

[Ollama](https://ollama.com) makes it easy to download and run open-weight models on your own machine. You don't have to use a specific model; choose from the Qwen, Gemma, Llama, Mistral, or other families as you see fit to best suit your needs and your hardware.

### Do some research first

Before downloading anything, spend a few minutes researching (by searching the web, or asking your LLM of choice) which model is the best fit for your laptop: one that runs in a reasonable amount of time and doesn't take up a ton of disk space. For most laptops this means a model in the roughly 12 to 24 GB range, or smaller depending on your hardware (see the tiers below).

This is time well spent: it gives you a relatively capable local model that you can run cost-free throughout the rest of the DSAN program for LLM-based processing tasks.

### Key ideas

Pick the largest model whose download is comfortably smaller than your RAM.

* **RAM is the limit.** The whole model must fit in memory (unified memory on Apple Silicon, VRAM on an NVIDIA GPU). Leave 6 to 8 GB free for the OS, your browser, and the model's context.
* **Quantization.** Weights are stored at reduced precision (4-bit is the usual default). A `q4_K_M` model is about a quarter the size of `bf16`, with a small quality loss. Default Ollama tags are already 4-bit.
* **Dense vs MoE.** A dense model uses every weight for every token. A mixture-of-experts (MoE) model, marked like `35b-a3b` (35B total, about 3B active), stores many experts but runs only a few per token, so it is much faster than a dense model of similar size.
* **MLX on Macs.** Tags ending in `-mlx` use Apple's MLX backend and are usually faster on Apple Silicon. On Windows, Linux, or Intel Macs, use the plain tags instead.
* **Speed matters as much as quality.** A slightly weaker model at 40 tokens/sec is often more useful day to day than a stronger one at 5 tokens/sec. You have about 900 job descriptions to process.

### Pick your tier

Find your machine's RAM, then start with the first model listed for your tier. Sizes are Ollama download sizes from [ollama.com](https://ollama.com/library) at the time of writing.

The last column gives a rough closed-weight (proprietary) model with comparable performance, to help you calibrate expectations. These are ballpark comparisons based on public benchmarks and model cards. Small open models often match older frontier models on reasoning benchmarks, but they know less about the world and make more mistakes on long, messy inputs, so treat the column as orientation, not a guarantee.

| Your RAM | Model | Type | Download | Mac (MLX) tag | Other platforms tag | Best for | Rough closed-weight analog |
|---|---|---|---|---|---|---|---|
| 8 GB | Qwen3.5 0.8B | Dense | ~1.2 GB | `qwen3.5:0.8b-mlx` | `qwen3.5:0.8b` | Very old or low-memory machines; quick tests | Roughly GPT-3 (2020); below GPT-3.5 |
| 8 GB | Qwen3.5 2B | Dense | ~2 to 3 GB | `qwen3.5:2b-mlx` | `qwen3.5:2b` | Simple extraction on a budget | Roughly GPT-3.5 Turbo (2023) |
| 8 GB | Qwen3.5 4B | Dense | ~3.3 to 4 GB | `qwen3.5:4b-mlx` | `qwen3.5:4b` | Light chat, quick tests | GPT-4o mini (2024) class |
| 16 GB | Gemma 4 12B | Dense | ~8 GB | `gemma4:12b-mlx` | `gemma4:12b` | Best all-round small model | GPT-4o (2024) class |
| 16 GB | Qwen3.5 9B | Dense | ~6.6 to 9 GB | `qwen3.5:9b-mlx` | `qwen3.5:9b` | Coding and reasoning at small size | GPT-4o (2024) class; below GPT-5 mini |
| 16 GB | Gemma 4 E4B | Dense, efficient | ~6.6 to 9.5 GB | `gemma4:e4b-mlx` | `gemma4:e4b` | Fastest small option; image input | GPT-4o mini (2024) class |
| 24 GB | gpt-oss 20B | MoE | ~14 GB | (no MLX tag) | `gpt-oss:20b` | Fast reasoning; OpenAI's open-weight model | o3-mini (early 2025) class |
| 24 to 32 GB+ | Gemma 4 26B-A4B | MoE | ~16 to 19 GB | `gemma4:26b-mlx` | `gemma4:26b` | Fast, strong, good with images | o3-mini / GPT-4.1 (2025) class |
| 32 GB+ | Qwen3.6 35B-A3B | MoE | ~24 GB | `qwen3.6:35b-mlx` | `qwen3.6:35b-a3b` | Best overall speed and quality | GPT-5 mini (2025) class; below GPT-5 |
| 32 GB+ | Qwen3.6 27B | Dense | ~17 to 19 GB | `qwen3.6:27b-mlx` | `qwen3.6:27b` | Highest quality; slower | GPT-5 mini (2025) class |
| 64 GB+ | gpt-oss 120B | MoE | ~65 GB | (no MLX tag) | `gpt-oss:120b` | Workstations only | o4-mini (2025) class |

Other families are also fine choices. A few that fit on typical laptops:

| Your RAM | Model | Type | Download | Tag (all platforms) | Rough closed-weight analog |
|---|---|---|---|---|---|
| 8 GB | Llama 3.2 3B | Dense | ~2 GB | `llama3.2:3b` | GPT-3.5 Turbo (2023) class |
| 16 GB | Ministral 3 8B | Dense | ~6 GB | `ministral-3:8b` | GPT-4o mini (2024) class |
| 16 GB | Ministral 3 14B | Dense | ~9 GB | `ministral-3:14b` | Between GPT-4o mini and GPT-4o |
| 24 to 32 GB | Mistral Small 3.2 24B | Dense | ~15 GB | `mistral-small3.2:24b` | GPT-4o mini (2024) class |

Notes:

* **Coding focus**: the Qwen3.6 models also come in `-coding` variants, for example `qwen3.6:35b-a3b-coding`.
* **Larger Llama models**: the newer Llama 4 models (e.g. `llama4:scout`, ~67 GB) are too large for most laptops.
* **This changes fast.** New open-weight models appear every few months; check the [Ollama model library](https://ollama.com/library) for newer releases.

### Pull, run, and manage models

Install Ollama from [ollama.com/download](https://ollama.com/download), then use these commands in a terminal.

```bash
# Download a model (example: Gemma 4 12B, MLX build for Apple Silicon)
ollama pull gemma4:12b-mlx

# Chat with it interactively (type /bye to exit)
ollama run gemma4:12b-mlx

# One-shot prompt from the command line
ollama run gemma4:12b-mlx "Explain gradient descent in three sentences."

# See what is installed and how big each model is
ollama list

# See what is loaded in memory right now
ollama ps

# Delete models you no longer need (several at once is fine)
ollama rm gemma4:12b-mlx qwen3.5:9b-mlx
```

* `ollama run` pulls the model automatically if it is not installed yet.
* Ollama also serves an API at `http://localhost:11434`, which is how you will call your local model from Python (see the starter code below).

### Tips and caveats

* **Check your RAM**: on a Mac, Apple menu > About This Mac; on Windows, Settings > System > About.
* **Running slow or swapping?** Drop one tier, or close memory-heavy apps. A model that barely fits will crawl.
* **Disk space**: models are large. Use `ollama list` and `ollama rm` to clean up.
* **Thinking models**: some models (e.g. Qwen3.5 / Qwen3.6) "think" before answering by default, which is slow. For simple extraction you can usually turn this off with `think=False` in the `ollama` Python client.
* **Local models are not frontier models.** They are great for private data, offline work, and experiments, but expect more mistakes than top hosted assistants. Verify outputs.

## Starter code

The `openai` Python package (`pip install openai`) can talk to OpenAI, to many other providers that offer an OpenAI-compatible endpoint (including Gemini), and to your local Ollama server. This means switching between Option A and Option B can be a small configuration change.

The snippet below is a sketch, not a solution. Adapt it to your provider and model, and check your provider's documentation for current model names and pricing.

In [ ]:
import json
import os

from openai import OpenAI
import pandas as pd

BACKEND = "local"   # "local" (Option B, Ollama) or "api" (Option A)

if BACKEND == "local":
    # Ollama must be running (open the Ollama app, or run `ollama serve`)
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # key is required but ignored
    MODEL = "qwen3.5:4b-mlx"   # the tag you pulled, e.g. from `ollama list`
else:
    with open(os.path.expanduser("~/.api-keys.json")) as f:
        keys = json.load(f)
    client = OpenAI(api_key=keys["openai"])
    MODEL = "your-model-name"   # a small, inexpensive model from your provider's model list

In [3]:
SYSTEM_PROMPT = """You extract structured information from job postings.
Return ONLY a JSON object with these keys:
job_title, company_name, city, state, job_type, remote_option,
salary_min, salary_max, experience_level, min_years_experience,
education, skills.
Use null for anything not stated in the posting. Do not guess."""


def extract_job_features(description):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": description},
        ],
        response_format={"type": "json_object"},   # ask for valid JSON
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)


# Try it on ONE job first, e.g.:
jobs = pd.read_json("data/Statistician-20261023-141624.json")

job = jobs["jobs_results"].iloc[0]
job_text = json.dumps(job, ensure_ascii=False)
extract_job_features(job_text)

KeyboardInterrupt: 

If you go the local route, the native `ollama` Python package (`pip install ollama`) is another option. It can enforce a JSON schema on the output, e.g. one generated from a `pydantic` model, which makes the results much more reliable to parse:

```python
import ollama
from pydantic import BaseModel

class JobFeatures(BaseModel):
    job_title: str | None
    company_name: str | None
    skills: list[str]
    # ... add the rest of your features
    city: str | None
    state: str | None
    country: str | None
    job_type: str | None
    remote_option: str | None
    salary_min: float | None
    salary_max: float | None
    salary_period: str | None
    experience_level: str | None
    min_years_experience: float | None
    education: str | None

response = ollama.chat(
    model="qwen3.5:0.8b",
    messages=[{"role": "user", "content": "Extract the features from this job posting:\n\n" + description}],
    format=JobFeatures.model_json_schema(),
    options={"temperature": 0},
)
features = JobFeatures.model_validate_json(response.message.content)
```

Most API providers offer a similar "structured outputs" feature; see their documentation.

## Workflow tips (both options)

* **Prototype on a handful of jobs** (5 to 10), inspect the outputs carefully against the original descriptions, and iterate on your prompt before processing everything.
* **Cache results to disk as you go** (e.g. one JSON line per job, keyed by `job_id`), so that a crash or a re-run does not repeat work you already paid for, in money or in time.
* **Validate and post-process.** LLM output still needs cleaning: check types, normalize categories, and handle missing or malformed responses.
* **Measure.** Time a few calls to estimate the total run time before launching the full job, and track tokens and cost on the API path.
* **Commit regularly**, e.g. after your prompt works, and again after the full run.

In [3]:
# INSERT CODE HERE

import ollama
from pydantic import BaseModel
from pathlib import Path
import glob
import os
import json

CACHE_PATH = Path("data/job_features_cache.jsonl")
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

def load_cached():
    cached = {}

    if CACHE_PATH.exists():
        with CACHE_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                cached[record["job_id"]] = record["features"]

    return cached


def append_cached(job_id, features):
    record = {
        "job_id": job_id,
        "features": features.model_dump(),
    }

    with CACHE_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

class JobFeatures(BaseModel):
    job_title: str | None
    company_name: str | None
    skills: list[str]
    benefits: list[str]
    city: str | None
    state: str | None
    country: str | None
    job_type: str | None
    remote_option: str | None
    salary_min: float | None
    salary_max: float | None
    salary_period: str | None
    experience_level: str | None
    min_years_experience: float | None
    education: str | None

def extract_job_features(description) -> JobFeatures:
    response = ollama.chat(
        model="qwen3.5:4b-mlx",
        messages=[{"role": "user", "content": "Extract the features from this job posting:\n\n" + description}],
        format=JobFeatures.model_json_schema(),
        think=False,
        options={"temperature": 0},
    )
    return JobFeatures.model_validate_json(response.message.content)

file_paths = sorted(
    glob.glob("data/*.json"),
    key=os.path.getmtime,
    reverse=True,
)[:90]

cached_results = load_cached()

for file_path in file_paths:
    try:
        with open(file_path, encoding="utf-8") as f:
            saved = json.load(f)
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as e:
        print(f"Skipping {file_path}: {e}")
        continue

    jobs = saved.get("jobs_results", [])

    for i, job in enumerate(jobs):
        job_id = job.get("job_id", f"row_{i}")

        if job_id in cached_results:
            print(f"Skipping cached job {job_id}")
            continue

        job_text = json.dumps(job, ensure_ascii=False)

        try:
            features = extract_job_features(job_text)
            append_cached(job_id, features)
            cached_results[job_id] = features.model_dump()
            print(f"Saved job {job_id}")
        except Exception as e:
            print(f"Failed on job {job_id}: {e}")
            continue

cache_df = pd.read_json(CACHE_PATH, lines=True)


features_df = pd.json_normalize(cache_df["features"])
features_df.insert(0, "job_id", cache_df["job_id"])
features_df.to_csv("data/processed-jobs-2.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(features_df)} extracted jobs")

Skipping cached job eyJqb2JfdGl0bGUiOiJEYXRhIEFyY2hpdGVjdCAoV2VlaGF3a2VuKSIsImNvbXBhbnlfbmFtZSI6IlRlY2hTb3VyY2UiLCJodGlkb2NpZCI6ImlSWFozXzc0OXYtTlZWRjhBQUFBQUE9PSIsInV1bGUiOiJ3K0NBSVFJQ0lOVlc1cGRHVmtJRk4wWVhSbGN3IiwiZ2wiOiJ1cyIsImhsIjoiZW4iLCJmYyI6IkVzc0JDb3dCUVVwcFZEUjBTMk52Y2xsamJVOXpSVzlrYjJNM1NuaFBjbFJvWlZBMVVHaFZOa05LVVY5M00yVk1WamcwYzNaR056UjVWMFptTlZWSWRVUktXVmRETFVWNlZrUjJVekU1UmxaWE5sRTNiREZhVkRodVptOWpabmRZY2t0ellUSmFOR3RHUTJKVFVHSk9kRnBxV25wQk5UQnFNbEYzV0doWVQxcEJaRFZDVW5kNlYyaGhUMVV3TlhBMmJqWVNGbTlMYm1KaGRreDZURjh6V1RWT2IxQnhOakpwUlVFYUlrRkVjM0k1WmxFNVNsVXdXRFZKVlVjeFMyNVpTWE4yZFVSemVtWmxjM2hMVUZFIiwiZmN2IjoiMyIsImZjX2lkIjoiaVJYWjNfNzQ5di1OVlZGOEFBQUFBQT09In0=
Skipping cached job eyJqb2JfdGl0bGUiOiJEYXRhIEFyY2hpdGVjdCIsImNvbXBhbnlfbmFtZSI6IktvY2ggRW5naW5lZXJlZCBTb2x1dGlvbnMiLCJodGlkb2NpZCI6ImpMLUlteHBnVmhESWk3R0xBQUFBQUE9PSIsInV1bGUiOiJ3K0NBSVFJQ0lOVlc1cGRHVmtJRk4wWVhSbGN3IiwiZ2wiOiJ1cyIsImhsIjoiZW4iLCJmYyI6IkVzc0JDb3dCUVVwcFZEUjBTMmh1UTJGUGFHNTNSVXcwY1RkU1lVOVlZeTFGY2tORVl

In [4]:



import pandas as pd 
cache_df = pd.read_json(CACHE_PATH, lines=True)


features_df = pd.json_normalize(cache_df["features"])
features_df.insert(0, "job_id", cache_df["job_id"])
features_df.to_csv("data/processed-jobs-2.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(features_df)} extracted jobs")

Saved 796 extracted jobs


## Reflection

At the end of the notebook, write a short paragraph (a few sentences each is fine) answering:

* Which option did you choose (API or local), which model, and why? If local, what hardware (RAM, chip/GPU)?
* How long did it take to process all of your job descriptions, and (for the API path) roughly how many tokens and dollars did it cost?
* How did the quality compare with your manual cleaning in HW-3.2? Where did the LLM do better, and where did it make mistakes?
* If you had to process 1 million job postings every month for a company, would you choose API or local compute? Explain the trade-offs.

## Your reflection

I chose the local because I already had it installed and it was free. My mac has a M2 chip and 8GB RAM. It took many hours to run mine since at first think was on. Even with think off it was very slow. The quality was better since so much of the data was merged into single columns that it was hard to process it uniformly. If I had to process 1 million job companies every month I would choose API since my computer would be so much slower and not be able to handle it. If a company is willing to pay that is preferred. The tradeoffs are privacy that comes with local and cost which is lower locally.